# PN1E: practical effectiveness of the informative third

## TL;DR

On prime-23 development data, using two prior ARA readings instead of one reduces held-out next-reading cross-entropy by `0.4742` bits, or `18.55%`. Exact-bin accuracy rises from `31.70%` to `41.71%`, and top-three accuracy from `69.46%` to `83.90%`.

An exact first-order raw-gap Markov projection produces `0.2738` bits of the `0.4742`-bit gross effect. The remaining `0.20048` bits are ordered structure beyond that control. This is a strong practical development result, not proof of exactly three waves. Prime 29 remains unopened.


## 1. Context and frozen method

For consecutive prime-wheel gaps,

\[x_i=\frac{2g_{i+1}}{g_i+g_{i+1}}\in(0,2).\]

The primary task predicts the next of 12 equal ARA bins. `ARA-Markov-1` uses the current ARA bin; `ARA-Markov-2` uses the previous and current bins. Models are trained on one consecutive half of the prime-23 cycle and scored on the other, then reversed.

Protocol SHA-256: `484B45190DCDC3823CDF6B2F644FCC87FCD925DA22B45321D2C334E56B8C77EB`.


## 2. Reproduce the primary analysis

This reconstructs all 36,495,360 circular prime-23 gaps, runs both cross-fit directions at 8, 12 and 16 bins, computes exact control entropies and contribution tables, and rewrites the canonical machine outputs.


In [1]:
import json
from pathlib import Path
import pandas as pd

import pn1e_third_memory_effectiveness as primary

HERE = Path.cwd()
primary.main()
results = json.loads((HERE / "PN1E_RESULTS.json").read_text(encoding="utf-8"))
print("\nPrimary classification:", results["primary_effectiveness"]["classification"])
print("Prime 29 opened:", results["data"]["prime29_opened"])


B=8 cross-fit complete
B=12 cross-fit complete
B=16 cross-fit complete
{
  "primary_effectiveness": {
    "classification": "STRONG PRACTICAL EFFECT",
    "scale": "bits per next 12-bin ARA reading",
    "markov1_cross_entropy_bits": 2.556497454238907,
    "markov2_cross_entropy_bits": 2.082283007411661,
    "gain_bits_per_reading": 0.4742144468272462,
    "relative_logloss_reduction": 0.18549380756900627,
    "perplexity_reduction": 0.2801413500221145,
    "markov1_top1_accuracy": 0.31697498717371053,
    "markov2_top1_accuracy": 0.4170699691215507,
    "markov1_top3_accuracy": 0.6946482177074804,
    "markov2_top3_accuracy": 0.838994281902607,
    "markov1_brier": 0.7895919235138439,
    "markov2_brier": 0.7030503267837246,
    "direction_gains_bits": [
      0.4741271557341391,
      0.4743017379203538
    ]
  },
  "control_scale": {
    "Empirical p23": {
      "one_neighbor_entropy_bits": 2.5564954578886283,
      "two_neighbor_entropy_bits": 2.0822464758983656,
      "memory_gain

## 3. Held-out predictive effectiveness

All metrics operate on the same next-ARA-reading task. Lower cross-entropy, perplexity and Brier score are better; higher top-one and top-three accuracy are better.


In [2]:
scores = pd.read_csv(HERE / "PN1E_EFFECTIVENESS_SCORES.csv")
primary_scores = scores[(scores["bins"] == 12) & (scores["direction"] == "mean")]
print(primary_scores.to_string(index=False, float_format=lambda value: f"{value:.9f}"))

m1 = primary_scores.loc[primary_scores["model"] == "ARA-Markov-1"].iloc[0]
m2 = primary_scores.loc[primary_scores["model"] == "ARA-Markov-2"].iloc[0]
print(f"\nCross-entropy gain: {m1['cross_entropy_bits_per_reading'] - m2['cross_entropy_bits_per_reading']:.9f} bits/read")
print(f"Relative uncertainty reduction: {(1 - m2['cross_entropy_bits_per_reading'] / m1['cross_entropy_bits_per_reading']):.4%}")
print(f"Exact-bin improvement: {(m2['top1_accuracy'] - m1['top1_accuracy']):.4%} points")
print(f"Top-three improvement: {(m2['top3_accuracy'] - m1['top3_accuracy']):.4%} points")


 bins direction        model  cross_entropy_bits_per_reading  perplexity  brier_score  top1_accuracy  top3_accuracy       observations
   12      mean      ARA-IID                     3.169739944 8.998845626  0.874152023    0.229148306    0.477749087 18247678.000000000
   12      mean ARA-Markov-1                     2.556497454 5.882777431  0.789591924    0.316974987    0.694648218 18247678.000000000
   12      mean ARA-Markov-2                     2.082283007 4.234768220  0.703050327    0.417069969    0.838994282 18247678.000000000

Cross-entropy gain: 0.474214447 bits/read
Relative uncertainty reduction: 18.5494%
Exact-bin improvement: 10.0095% points
Top-three improvement: 14.4346% points


![PN1E practical-effect diagnostics](PN1E_EFFECTIVENESS_DIAGNOSTIC.png)

The two-memory model wins in both held-out directions and at every predeclared resolution.


## 4. Relational scale of the controls

The raw-gap Markov control is not the one-memory ARA predictor. It is a hypothetical raw-gap generator that retains only immediate `current gap -> next gap` tendencies, then projects those sequences onto the same three-reading, 12-bin ARA task.


In [3]:
scale = pd.read_csv(HERE / "PN1E_ENTROPY_SCALE.csv")
print(scale.to_string(index=False, float_format=lambda value: f"{value:.9f}"))

empirical = scale.loc[scale["model"] == "Empirical p23"].iloc[0]
control = scale.loc[scale["model"] == "First-order gap Markov"].iloc[0]
excess = empirical["memory_gain_bits"] - control["memory_gain_bits"]
print(f"\nExcess above raw-gap Markov control: {excess:.9f} bits/read")
print(f"Fraction of gross gain above control: {excess / empirical['memory_gain_bits']:.4%}")


                 model  one_neighbor_entropy_bits  two_neighbor_entropy_bits  memory_gain_bits  one_neighbor_uncertainty_removed_fraction  one_neighbor_perplexity  two_neighbor_perplexity  perplexity_reduction_fraction
         Empirical p23                2.556495458                2.082246476       0.474248982                                0.185507461              5.882769290              4.234660984                    0.280158583
       IID-gap overlap                2.794344738                2.657568974       0.136775764                                0.048947348              6.937157973              6.309689336                    0.090450389
First-order gap Markov                2.608985290                2.335215984       0.273769307                                0.104933250              6.100744410              5.046265047                    0.172844376

Excess above raw-gap Markov control: 0.200479675 bits/read
Fraction of gross gain above control: 42.2731%


## 5. Attribution

The third-reading benefit is distributed across a nonlinear web. The top five ARA contexts account for only `22.53%` of total conditional information and the top twenty for `57.83%`.


In [4]:
contexts = pd.read_csv(HERE / "PN1E_CONTEXT_ATTRIBUTION.csv")
raw_top = pd.read_csv(HERE / "PN1E_TOP30_GAP_QUADRUPLES.csv")
print("Top ten ARA contexts:")
print(contexts.head(10).to_string(index=False, float_format=lambda value: f"{value:.9f}"))
print("\nTop ten raw four-gap constellations:")
print(raw_top.head(10).to_string(index=False, float_format=lambda value: f"{value:.9f}"))


Top ten ARA contexts:
 rank  first_context_bin  second_context_bin  first_context_center  second_context_center  context_probability  contribution_bits_per_reading  dominant_next_bin_markov2  dominant_next_center_markov2  dominant_next_bin_markov1  dominant_next_center_markov1  top_prediction_changes
    1                  4                   4           0.750000000            0.750000000          0.053520530                    0.028940290                          8                   1.416666667                          8                   1.416666667                   False
    2                  8                   8           1.416666667            1.416666667          0.014973191                    0.022382606                          5                   0.916666667                          7                   1.250000000                    True
    3                  7                   8           1.250000000            1.416666667          0.011079189                    0.019807

## 6. Independent validation

The standalone validator independently reconstructs the gap cycle, relation encoding, cross-fit scores, entropy controls and contribution sums. It does not import the primary PN1E module.


In [5]:
import pn1e_independent_validator as validator

validator.main()
audit = json.loads((HERE / "PN1E_INDEPENDENT_VALIDATION.json").read_text(encoding="utf-8"))
print("\nAll independent checks pass:", audit["all_checks_pass"])
print("Maximum primary-score absolute error:", audit["maximum_primary_score_absolute_error"])


{
  "protocol": "PN1E/DEV/v1",
  "independent_route": "standalone p23 construction, vectorized relation encoding, direct held-out categorical scoring, event-count raw attribution",
  "prime29_opened": false,
  "checks": {
    "protocol_hash_exact": true,
    "prime29_remains_unopened": true,
    "gap_count_exact": true,
    "gap_period_exact": true,
    "gap_hash_exact": true,
    "all_primary_scores_reproduced": true,
    "empirical_cmi_reproduced": true,
    "context_sum_equals_cmi": true,
    "raw_sum_equals_cmi": true,
    "positive_raw_mass_reproduced": true,
    "negative_raw_mass_reproduced": true,
    "top_quadruple_reproduced": true
  },
  "all_checks_pass": true,
  "maximum_primary_score_absolute_error": 7.105427357601002e-15,
  "empirical_cmi_bits": 0.47424898199026283,
  "raw_attribution": {
    "positive_bits": 0.625829292460832,
    "negative_bits": -0.15158031047056728,
    "net_bits": 0.47424898199026466,
    "top_quadruple": [
      2,
      4,
      8,
      6
    ],


## 7. Takeaways

The informative third is operationally useful: arrival path materially improves prediction of the next ARA position. About `57.7%` of the gross information gain is reproduced by a first-order raw-gap transition world, while `42.3%` remains above that control.

The next development branch should decompose the full two-reading state into direction, distance and raw child identity before freezing a transfer model for unopened prime 29.

## Provenance

- Protocol: `PN1E_THIRD_MEMORY_EFFECTIVENESS_PROTOCOL.md`
- Primary analysis: `pn1e_third_memory_effectiveness.py`
- Independent audit: `pn1e_independent_validator.py`
- Full report: `PN1E_THIRD_MEMORY_EFFECTIVENESS_REPORT.md`
